# Image Target Selection

This notebook load the `v1` Pa/Br emitting galaxy catalog and down-select the sampled based on **imaging-only** criteria (i.e. whether the broadband image is good for weak lensing measurement). A KL sample first has to be a WL sample.

- No blending/merging galaxies
- No bleeding
- No bad pixels
- Star-galaxy separation shows it's an extended galaxy, not point source like star or AGN
- Not on the edge of detector
- spatial resolution factor R > 0.4


In [1]:
import numpy as np
import astropy.io.fits as fits
from astropy.table import Table
import matplotlib.pyplot as plt
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.nddata import Cutout2D, NoOverlapError, NDData
from astropy.visualization import ZScaleInterval
from astropy import wcs, table

In [2]:
hdul = fits.open("/xdisk/timeifler/jiachuanxu/jwst/jades/catalogs/hlsp_jades_jwst_nircam_goods-n_photometry_v5.0_catalog.fits")

In [3]:
hdul.info()

Filename: /xdisk/timeifler/jiachuanxu/jwst/jades/catalogs/hlsp_jades_jwst_nircam_goods-n_photometry_v5.0_catalog.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU      63   ()      
  1  FILTERS       1 BinTableHDU     85   27R x 12C   [6A, E, E, E, E, E, E, E, E, E, E, E]   
  2  FLAG          1 BinTableHDU    279   181144R x 90C   [K, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, J, J, J]   
  3  SIZE          1 BinTableHDU    103   181144R x 18C   [J, D, D, D, D, D, D, K, K, K, K, D, D, D, D, D, D, D]   
  4  CIRC          1 BinTableHDU   1631   181144R x 528C   [J, D, D, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E

In [11]:
Table(hdul["FLAG"].data)

ID,RA,DEC,F070W_FLAG,F070W_WHT,F070W_TEXP,F090W_FLAG,F090W_WHT,F090W_TEXP,F115W_FLAG,F115W_WHT,F115W_TEXP,F150W_FLAG,F150W_WHT,F150W_TEXP,F162M_FLAG,F162M_WHT,F162M_TEXP,F182M_FLAG,F182M_WHT,F182M_TEXP,F187N_FLAG,F187N_WHT,F187N_TEXP,F200W_FLAG,F200W_WHT,F200W_TEXP,F210M_FLAG,F210M_WHT,F210M_TEXP,F277W_FLAG,F277W_WHT,F277W_TEXP,F300M_FLAG,F300M_WHT,F300M_TEXP,F335M_FLAG,F335M_WHT,F335M_TEXP,F356W_FLAG,F356W_WHT,F356W_TEXP,F410M_FLAG,F410M_WHT,F410M_TEXP,F430M_FLAG,F430M_WHT,F430M_TEXP,F444W_FLAG,F444W_WHT,F444W_TEXP,F460M_FLAG,F460M_WHT,F460M_TEXP,F435W_FLAG,F435W_WHT,F435W_TEXP,F606W_FLAG,F606W_WHT,F606W_TEXP,F775W_FLAG,F775W_WHT,F775W_TEXP,F814W_FLAG,F814W_WHT,F814W_TEXP,F850LP_FLAG,F850LP_WHT,F850LP_TEXP,F105W_FLAG,F105W_WHT,F105W_TEXP,F125W_FLAG,F125W_WHT,F125W_TEXP,F140W_FLAG,F140W_WHT,F140W_TEXP,F160W_FLAG,F160W_WHT,F160W_TEXP,F770W_FLAG,F770W_WHT,F770W_TEXP,F1280W_FLAG,F1280W_WHT,F1280W_TEXP,FLAG_BN,PARENT_ID,PID_HASH
int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,int32,int32,int32
1000003,189.13002,62.21184,992.0,0.0,0.0,0.0,49598.90234375,21883.853515625,0.0,15838.8828125,10217.5947265625,0.0,4621.185546875,1030.730224609375,992.0,0.0,0.0,0.0,8784.1630859375,5099.59619140625,992.0,0.0,0.0,0.0,21767.830078125,4378.4912109375,0.0,5360.15478515625,3821.64013671875,0.0,50155.04296875,1028.132568359375,992.0,0.0,0.0,0.0,15532.7705078125,1028.652099609375,0.0,101425.109375,3501.90478515625,0.0,12533.751953125,1028.132568359375,992.0,0.0,0.0,0.0,91650.265625,5547.033203125,992.0,0.0,0.0,0.0,704122.4375,17062.51953125,0.0,242976.5625,11558.6689453125,0.0,654544.125,22148.4375,0.0,618909.5625,24961.9296875,0.0,1794446.125,53812.92578125,0.0,14454.4404296875,18457.88671875,0.0,8493.1416015625,12527.80078125,0.0,479.0129089355469,990.4063720703125,0.0,9892.56640625,13676.986328125,992.0,0.0,0.0,992.0,0.0,0.0,0,1000003,4229128
1000011,189.12532,62.212246,1680.0,0.0,0.0,0.0,49487.98046875,20950.80078125,0.0,15495.546875,8971.875,0.0,4768.68701171875,1030.730224609375,1680.0,0.0,0.0,0.0,10619.29296875,6517.47265625,1680.0,0.0,0.0,0.0,21879.888671875,4375.9814453125,0.0,6820.173828125,5282.490234375,0.0,49182.171875,1028.582763671875,1680.0,0.0,0.0,0.0,14521.046875,1027.6624755859375,0.0,100645.03125,3492.144287109375,0.0,12370.8955078125,1026.7421875,1680.0,0.0,0.0,0.0,96240.515625,5917.396484375,1680.0,0.0,0.0,0.0,710786.8125,17224.013671875,0.0,228327.359375,10861.7900390625,0.0,505821.9375,17115.98046875,0.0,614671.5625,24791.0,0.0,1577524.75,47307.76171875,0.0,14278.0546875,18232.6484375,0.0,8821.5673828125,13012.244140625,0.0,486.0794677734375,1005.0172729492188,0.0,9464.275390625,13084.8525390625,1680.0,0.0,0.0,1680.0,0.0,0.0,0,1000011,4229128
1000015,189.12782,62.212456,837.0,0.0,0.0,0.0,51411.87890625,22900.12890625,0.0,18551.328125,11921.7158203125,0.0,6872.49365234375,2033.75244140625,837.0,0.0,0.0,0.0,11644.46484375,8416.0712890625,837.0,0.0,0.0,0.0,25021.3359375,5389.78125,0.0,7609.310546875,6862.39794921875,0.0,53565.5546875,1396.4727783203125,837.0,0.0,0.0,0.0,19130.1640625,1381.6953125,0.0,110936.96875,3867.466552734375,0.0,14500.53515625,1400.782958984375,837.0,0.0,0.0,0.0,97778.390625,6258.77880859375,837.0,0.0,0.0,0.0,704774.0625,17078.310546875,0.0,243994.765625,11607.1064453125,0.0,555962.9375,18812.650390625,0.0,660068.6875,26621.9609375,0.0,1651069.375,49513.2578125,0.0,14612